# USE THIS WITH DEFAULT ADB DATA

### Load the saved model

In [ ]:
import pickle
import pyarrow.dataset as ds
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GroupShuffleSplit
import pickle

short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'DATETIME',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

WINDOW_SIZE = 4
model = pickle.load(open("../data/models/test_model.sav", 'rb'))

In [ ]:
import pandas as pd

def setup_input(df, window_size = 4, output_path=None):
    """
    Optimized function to create a sliding window dataset.
    
    Args:
    - df (pd.DataFrame): Input DataFrame.
    - window_size (int): Size of the sliding window.
    - output_path (str): Path to save the output. If None, returns a DataFrame.
    
    Returns:
    - pd.DataFrame or None: Processed DataFrame if `output_path` is None.
    """
    # Create a generator to yield rows for each window
    def generate_rows():
        for policy, group in df.groupby('POLICY'):
            # Convert columns to NumPy arrays for efficient slicing
            flows = group['FLOW'].to_numpy()
            technology = group['TECHNOLOGY'].to_numpy()
            usage = group['USAGE'].to_numpy()
            housing = group['HOUSING'].to_numpy()
            consumption = group['CONSUMPTION'].to_numpy()
            hour_date = group['DATETIME'].to_numpy()
            weekday = group['WEEKDAY'].to_numpy()
            hours = group['HOUR'].to_numpy()

            # Generate sliding windows
            for i in range(len(flows) - window_size + 1):
                row = {
                    'POLICY': policy,
                    'TECHNOLOGY': technology[i],
                    'USAGE': usage[i],
                    'HOUSING': housing[i],
                    'CONSUMPTION': consumption[i],
                    'DATETIME': hour_date[i],
                    'WEEKDAY': weekday[i],
                    'START_HOUR': hours[i],
                }
                # Add flow values for the window
                for j in range(window_size):
                    row[f'FLOW_{j + 1}'] = flows[i + j]
                yield row

    # Write to file or return as DataFrame
    if output_path:
        pd.DataFrame.from_records(generate_rows()).to_parquet(output_path, index=False)
        print(f"Saved processed data to {output_path}")
        return None
    else:
        return pd.DataFrame.from_records(generate_rows())


### Load and format the data

In [ ]:
file_path = '../data/splits/split_25.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()
df = df.rename(columns=short_names)

df['DATETIME'] = pd.to_datetime(df['DATETIME'])
df['WEEKDAY'] = df['DATETIME'].dt.dayofweek  # Extract weekday (Monday=0, Sunday=6)
df['HOUR'] = df['DATETIME'].dt.hour

df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
df = df.dropna(subset=["FLOW"])

df_set = setup_input(df)

# Dynamically construct the FLOW column names
flow_columns = [f"FLOW_{i}" for i in range(1, WINDOW_SIZE + 1)]
# Define the full feature list
base_features = ['USAGE', 'HOUSING', 'WEEKDAY', 'START_HOUR']
features = base_features + flow_columns

# Select features and target
X = df_set[features]
label_encoders = {}

for col in X.select_dtypes(include='object').columns:  # Select categorical columns
    le = LabelEncoder()
    X.loc[:, col] = le.fit_transform(X[col])  # Use .loc[] for explicit assignment
    label_encoders[col] = le

In [ ]:
y = model.predict(X)

In [ ]:
df_set['LEAK'] = y

In [ ]:
counts = df_set['LEAK'].value_counts()
print(counts)

In [ ]:
df_set